# Get to Know a Dataset: GeoNadir FAIR Data

This notebook is a guided tour of the [GeoNadir FAIR Data](https://registry.opendata.aws/geonadir-fair-data) 
collection: open UAV (drone) survey datasets contributed through GeoNadir and published as cloud-native 
geospatial files. You can also browse the collection in your browser at 
[https://data.geonadir.com/fairgeo](https://data.geonadir.com/fairgeo). More usage examples, tutorials, and documentation for this dataset and others can 
be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

> **Access method 1 of 2: plain AWS S3.** This notebook uses **only Amazon S3** — `boto3` to list and read objects, Rasterio to read the Cloud Optimized GeoTIFFs — and nothing else. No STAC client and no extra endpoint, so it runs as soon as the data is uploaded to the public bucket.
>
> The companion notebook, `get-to-know-a-dataset-stac.ipynb`, is **access method 2 of 2: the STAC catalog** — richer search by area, date, and habitat, once the STAC endpoint is live.

## Understanding the dataset structure

GeoNadir FAIR Data is organized so the files are simple to address directly. Each UAV survey is one 
folder under the `datasets/` prefix, keyed by a stable public item id. Inside a folder, every file is 
named `{dataset_id}-{file_type}.{ext}` — the numeric dataset id, a file-type token, and an extension:

```text
s3://geonadir-fair-data/
  stac/                                 <- STAC catalog (optional discovery layer)
  datasets/
    {item_id}/                          <- item_id = {dataset_id}-{uav_uid}
      {dataset_id}-metadata.json        (required)
      {dataset_id}-raw_images.zip       (required)
      {dataset_id}-rgb_ortho.tif        (optional, Cloud Optimized GeoTIFF)
      {dataset_id}-dsm.tif              (optional, Cloud Optimized GeoTIFF)
      {dataset_id}-dtm.tif              (optional, Cloud Optimized GeoTIFF)
      {dataset_id}-multispec_ortho.tif  (optional, Cloud Optimized GeoTIFF)
```

One UAV survey = one `datasets/{item_id}/` folder, and the item id itself is `{dataset_id}-{uav_uid}`. 
Only the files that exist for a survey are present, so a survey with no DTM simply has no 
`{dataset_id}-dtm.tif`.

This notebook talks to the bucket directly with `boto3`, so you can see exactly how the files are laid out. (For search across many surveys by date/area/habitat, see the STAC companion notebook.)

In [ ]:
# CODING GUIDELINES (from the AWS template):
# This is a 101-level introduction using common tools and libraries.
# The bucket is public, so all S3 reads are UNSIGNED (no AWS account or credentials needed).

First, the libraries. This notebook talks to S3 directly with `boto3` (list + read objects) and reads the COGs with Rasterio. The dataset is public, so all S3 reads are unsigned — no AWS account or credentials required.

In [ ]:
# Install the required libraries into this kernel.
# On Colab, boto3/numpy/matplotlib are preinstalled but rasterio (GDAL) is NOT —
# so this cell is what makes the notebook run there. Safe to re-run; skip it if
# your environment already has these.
%pip install -q "boto3>=1.34" "rasterio>=1.3" "matplotlib>=3.8" "numpy>=1.26"

In [ ]:
# This notebook requires the following additional libraries
# (install with pip or conda for your environment):
#
# boto3 >= 1.34
# rasterio >= 1.3
# matplotlib >= 3.8
# numpy >= 1.26

# Built-ins
import json
from pprint import pprint

# Installed libraries
import boto3
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import rasterio.session
from botocore import UNSIGNED
from botocore.config import Config

Next, the dataset location.

In [ ]:
# Dataset location
BUCKET = "geonadir-fair-data"
REGION = "ap-southeast-2"

# Public bucket -> unsigned S3 requests (no credentials required)
s3 = boto3.client('s3', region_name=REGION, config=Config(signature_version=UNSIGNED))
aws_session = rasterio.session.AWSSession(aws_unsigned=True)  # for reading COGs from s3://

Without a catalog, we discover surveys by listing the `datasets/` prefix. Each returned prefix is one survey id.

In [ ]:
# List survey ids under datasets/
def list_survey_ids(max_ids=None):
    ids, token = [], None
    while True:
        kw = dict(Bucket=BUCKET, Prefix='datasets/', Delimiter='/')
        if token: kw['ContinuationToken'] = token
        resp = s3.list_objects_v2(**kw)
        for p in resp.get('CommonPrefixes', []):
            ids.append(p['Prefix'].split('/')[1])
            if max_ids and len(ids) >= max_ids:
                return ids
        if not resp.get('IsTruncated'):
            return ids
        token = resp['NextContinuationToken']

survey_ids = list_survey_ids(max_ids=10)
print(f"First {len(survey_ids)} survey id(s):")
for sid in survey_ids:
    print(sid)

To *filter* (by date, area, or habitat) without STAC, read each survey's small `metadata.json` and filter in Python. That is fine for a modest number of surveys; the STAC companion notebook does this efficiently server-side once the catalog is live.

In [ ]:
# Read a survey's metadata.json
# Files inside datasets/{item_id}/ follow the GeoNadir convention {dataset_id}-{file_type}.{ext},
# where dataset_id is the numeric id at the front of the item id (item_id = {dataset_id}-{uav_uid}).
def dataset_key(item_id, file_type):
    dataset_id = item_id.split('-', 1)[0]
    return f'datasets/{item_id}/{dataset_id}-{file_type}'

def read_metadata(item_id):
    key = dataset_key(item_id, 'metadata.json')
    with s3.get_object(Bucket=BUCKET, Key=key)['Body'] as body:
        return json.load(body)

def try_read_metadata(item_id):
    # Tolerant read: returns None for a folder without a readable metadata.json
    # (e.g. a partially-uploaded survey), so aggregation loops can skip it.
    try:
        return read_metadata(item_id)
    except Exception:
        return None

def habitats(metadata):
    # geonadir:iucn_habitat is a list (a survey can span several habitats)
    props = metadata.get('properties', metadata)
    h = props.get('geonadir:iucn_habitat') or []
    return [h] if isinstance(h, str) else list(h)

# Example: keep only Forest surveys among the first listed ids
forest = []
for sid in survey_ids:
    m = try_read_metadata(sid)
    if m is None:
        continue
    if 'Forest' in habitats(m):
        forest.append(sid)
print("Forest surveys:", forest)

## Data formats

GeoNadir FAIR Data uses a small, deliberately standard set of formats so the data works with off-the-shelf tools:

| Asset | Format | Why this format |
| --- | --- | --- |
| `{dataset_id}-metadata.json` | JSON | Normalized survey metadata; readable in any language |
| `{dataset_id}-raw_images.zip` | ZIP of JPEG | Original UAV frames kept together for reprocessing / photogrammetry |
| `{dataset_id}-rgb_ortho.tif`, `{dataset_id}-dsm.tif`, `{dataset_id}-dtm.tif`, `{dataset_id}-multispec_ortho.tif` | Cloud Optimized GeoTIFF (COG) | Read a window or overview **directly from S3** without downloading the whole file |

The COG is the key choice. A Cloud Optimized GeoTIFF is a normal GeoTIFF with internal tiling + overviews arranged so an HTTP range-request pulls only the pixels you ask for. GDAL, Rasterio, QGIS, and cloud-native pipelines can open a multi-gigabyte orthomosaic and read a small window or a low-res overview in seconds. Each COG carries its own CRS and geotransform, so projection metadata is read straight from the file. Bands follow the common EO convention (`B1`, `B2`, ... with `common_name` like `red`, `green`, `blue`, `rededge`, `nir`).

## Loading data from a survey

First the small `metadata.json` for one survey, then — the interesting part — opening an orthomosaic COG and reading only a low-resolution overview, straight from S3, without downloading the full GeoTIFF.

In [ ]:
# Pick the first listed survey that has a readable metadata.json
item_id = next((sid for sid in survey_ids if try_read_metadata(sid) is not None), survey_ids[0])
metadata = read_metadata(item_id)
pprint(metadata)

Now the orthomosaic. We open it with Rasterio over `s3://` (GDAL's `/vsis3/`). Because it is a COG, reading a decimated overview transfers only a few tiles — not the whole file.

In [ ]:
# Build the orthomosaic href from the key convention
ortho_href = f's3://{BUCKET}/' + dataset_key(item_id, 'rgb_ortho.tif')

with rasterio.Env(session=aws_session):
    with rasterio.open(ortho_href) as src:
        print('CRS:', src.crs)
        print('Size (WxH):', src.width, 'x', src.height)
        print('Bands:', src.count)
        print('Resolution (CRS units, e.g. m or degrees):', src.res)
        print('Bounds:', src.bounds)
        # Read a downsampled overview: at most ~1024 px on the long edge
        scale = max(1, max(src.width, src.height) // 1024)
        out_shape = (src.count, src.height // scale, src.width // scale)
        overview = src.read(out_shape=out_shape)
print('Overview array shape (bands, rows, cols):', overview.shape)

## Visualizing a survey

Two visuals. First, the RGB orthomosaic overview we just read — the survey as an image. Second, a collection-level view: where the surveys are, colored by IUCN habitat.

In [ ]:
# Display the RGB orthomosaic overview
rgb = np.transpose(overview[:3], (1, 2, 0)).astype('float32')  # first 3 bands: R,G,B
rgb = (rgb - rgb.min()) / (np.ptp(rgb) + 1e-9)                 # normalize to 0-1 for display

plt.figure(figsize=(9, 9), dpi=100)
plt.imshow(rgb)
plt.title(f'GeoNadir FAIR Data orthomosaic overview: {item_id}', fontsize=13, fontweight='bold')
plt.axis('off')
plt.show()

### Quick preview from the lowest COG overview

A COG bakes in a pyramid of ever-smaller overviews — like the zoom levels of a web map stored inside the file. The **lowest (coarsest)** level is a tiny thumbnail: reading it pulls only a few kilobytes, the cheapest possible way to eyeball a survey before committing to a bigger read.

In [ ]:
# Load an RGB preview from the LOWEST (coarsest) COG overview.
# The coarsest overview is a tiny thumbnail — a few KB over the wire — so this is
# the cheapest possible look at a survey.
with rasterio.Env(session=aws_session):
    with rasterio.open(ortho_href) as src:
        factors = src.overviews(1)          # decimation factors, e.g. [2, 4, 8, 16, 32]
        print('COG overview factors:', factors or '(none — file has no pyramid)')
        if factors:
            f = factors[-1]                 # last = coarsest = lowest level
            out_h = -(-src.height // f)     # ceil div -> the overview's exact size
            out_w = -(-src.width  // f)
        else:
            f, out_h, out_w = 1, src.height, src.width
        # Reading at the overview's own size makes GDAL pull straight from that level
        preview = src.read([1, 2, 3], out_shape=(3, out_h, out_w))  # R, G, B
print(f'Lowest overview: 1/{f} scale, {out_w} x {out_h} px')

rgb = np.transpose(preview, (1, 2, 0)).astype('float32')
rgb = (rgb - rgb.min()) / (np.ptp(rgb) + 1e-9)        # normalize to 0-1 for display
plt.figure(figsize=(5, 5), dpi=100)
plt.imshow(rgb)
plt.title(f'Lowest-overview RGB preview: {item_id}', fontsize=11, fontweight='bold')
plt.axis('off'); plt.show()

For the collection footprint, we read every survey's `metadata.json` once (in parallel — a few minutes for the whole collection), gather each survey's bounding-box centroid and habitat, then scatter them on a simple map — the kind of overview that tells a new user what the collection actually covers.

In [ ]:
# Map published surveys by centroid, colored by IUCN habitat.
# Reads every survey's metadata.json once, in parallel, and keeps them in
# all_metadata so later cells can reuse them without re-reading S3.
from concurrent.futures import ThreadPoolExecutor

all_ids = list_survey_ids()
with ThreadPoolExecutor(max_workers=32) as pool:
    all_metadata = {sid: m for sid, m in zip(all_ids, pool.map(try_read_metadata, all_ids))
                    if m is not None}  # skip folders without a readable metadata.json
print(f"Read metadata for {len(all_metadata)} of {len(all_ids)} surveys")

records = []
for sid, m in all_metadata.items():
    bbox = m.get('bbox')
    if not bbox:
        continue
    # a survey tagged with several habitats is plotted under its first one
    records.append(((bbox[0]+bbox[2])/2, (bbox[1]+bbox[3])/2,
                    (habitats(m) or ['Unknown'])[0]))

In [ ]:
# Plot the centroids
lons = [r[0] for r in records]; lats = [r[1] for r in records]; habitats = [r[2] for r in records]
cats = sorted(set(habitats))
cmap = {c: plt.cm.tab20(i % 20) for i, c in enumerate(cats)}
plt.figure(figsize=(12, 7), dpi=100)
for c in cats:
    xs = [lo for lo, h in zip(lons, habitats) if h == c]
    ys = [la for la, h in zip(lats, habitats) if h == c]
    plt.scatter(xs, ys, label=c, s=40, alpha=0.8, color=cmap[c])
plt.title('GeoNadir FAIR Data — published surveys by IUCN habitat', fontsize=14, fontweight='bold')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.grid(True, linestyle='--', alpha=0.3); plt.tight_layout(); plt.show()

## Worked example: measuring vegetation cover

**How much of a surveyed site is vegetated?** For a multispectral survey we compute NDVI (Normalized Difference Vegetation Index) directly from the red and near-infrared bands (located by their `eo:common_name` in the metadata, since band order differs between sensors) of the multispectral orthomosaic, then report the vegetated fraction of the survey. A common first-pass measure for habitat and land-cover work — and it runs off the COG without downloading the full raster.

In [ ]:
# Compute NDVI from a multispectral orthomosaic and report vegetated fraction.
# Not every survey has a multispectral orthomosaic, so pick one that does (prefer the current item).
# Band order differs between sensors, so find red and nir by their eo:common_name
# in the asset's band list rather than assuming fixed positions.
def band_index(asset, common_name):
    for i, b in enumerate(asset.get('bands', [])):
        if b.get('eo:common_name') == common_name:
            return i  # 0-based position in the array rasterio returns
    return None

def ndvi_bands(m):
    asset = m.get('assets', {}).get('multispec_ortho')
    if not asset:
        return None
    red, nir = band_index(asset, 'red'), band_index(asset, 'nir')
    return (red, nir) if red is not None and nir is not None else None

ms_item = item_id if ndvi_bands(all_metadata.get(item_id, {})) else next(
    (sid for sid, m in all_metadata.items() if ndvi_bands(m)), None)

if ms_item is None:
    print('No survey has a multispectral orthomosaic with red + nir bands — skipping NDVI.')
else:
    red_i, nir_i = ndvi_bands(all_metadata[ms_item])
    ms_href = f's3://{BUCKET}/' + dataset_key(ms_item, 'multispec_ortho.tif')
    with rasterio.Env(session=aws_session):
        with rasterio.open(ms_href) as src:
            scale = max(1, max(src.width, src.height) // 2048)
            out_shape = (src.count, src.height // scale, src.width // scale)
            arr = src.read(out_shape=out_shape).astype('float32')

    print(f'{ms_item}: red = B{red_i + 1}, nir = B{nir_i + 1}')
    red = arr[red_i]
    nir = arr[nir_i]
    ndvi = (nir - red) / (nir + red + 1e-9)
    valid = np.isfinite(ndvi) & (arr.sum(axis=0) > 0)
    vegetated = ndvi > 0.3            # common threshold for healthy vegetation
    frac = vegetated[valid].mean() if valid.any() else float('nan')
    print(f'Vegetated fraction of {ms_item} (NDVI > 0.3): {frac:.1%}')

    # viridis: perceptually uniform and colour-blind friendly (avoids the red-green
    # RdYlGn ramp, which is hard to read for red-green colour vision deficiency).
    plt.figure(figsize=(9, 8), dpi=100)
    plt.imshow(np.where(valid, ndvi, np.nan), cmap='viridis', vmin=-0.2, vmax=0.9)
    plt.colorbar(label='NDVI')
    plt.title(f'NDVI — {ms_item}', fontsize=13, fontweight='bold')
    plt.axis('off'); plt.show()

## Community challenge

**Habitat-scale change and classification from citizen UAV surveys.**

GeoNadir FAIR Data is a growing, globally distributed collection of centimetre-scale UAV surveys tagged with IUCN habitat classes. That combination is unusual: open satellite data is plentiful but coarse; centimetre-scale drone imagery with open licensing and consistent metadata is rare. Some questions we would love the community to take on:

1. **Habitat classification transfer.** Train a land-cover / habitat classifier on the multispectral + RGB orthomosaics and test how well it generalizes across IUCN habitat classes and geographies.
2. **Super-resolution priors for satellite data.** Use the centimetre-scale orthomosaics as ground truth to train or evaluate super-resolution / downscaling of Sentinel-2 or Landsat over the same footprints.
3. **Structure from the DSM/DTM pair.** Where both exist, derive canopy height models and study vegetation structure against the recorded habitat class.

**Getting started.** List `datasets/`, read each `metadata.json`, and group by `geonadir:iucn_habitat` (a list — a survey can carry several habitats) to assemble a per-habitat training set. Read overviews from the COGs first (fast, cheap) before committing to full-resolution reads, and treat the `providers` entries as uploader-supplied attribution rather than verified authority records. Start with a toy subset of a handful of surveys — a working small example beats a stalled large one.

---
*Contributions and derived tutorials are welcome — open a pull request against 
[open-data-examples](https://github.com/Geonadir/open-data-examples) or add your tutorial to the 
GeoNadir FAIR Data entry on the Registry of Open Data.*